In [0]:
"""NeuroPlex OpenTargets Ingestion Task.

Scheduled task for Lakeflow Job. Fetches target-disease associations
and tractability data for neuroscience targets.
"""
import sys, time, json, requests
from datetime import datetime, timezone
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import functions as F

OT_URL = "https://api.platform.opentargets.org/api/v4/graphql"
from config.neuroplex_config import load_config
CFG = load_config()
TABLE = CFG.prefixed_fqn("opentargets")

TARGET_GENES = [
    "HCRT", "HCRTR1", "HCRTR2",
    "PSEN1", "PSEN2", "APP", "MAPT", "GRN", "TARDBP",
    "SOD1", "FUS", "TBK1", "CHD8", "SCN2A", "SYNGAP1",
]

SEARCH_Q = """
query SearchTarget($queryString: String!, $size: Int!) {
  search(queryString: $queryString, entityNames: ["target"], page: {size: $size, index: 0}) {
    total
    hits { id entity name }
  }
}
"""

TARGET_Q = """
query TargetAssoc($ensemblId: String!, $size: Int!) {
  target(ensemblId: $ensemblId) {
    id
    approvedSymbol
    approvedName
    biotype
    tractability { label modality value }
    associatedDiseases(page: {size: $size, index: 0}) {
      count
      rows {
        disease { id name therapeuticAreas { id name } }
        score
        datasourceScores { id score }
      }
    }
  }
}
"""

def resolve_ensembl(gene):
    resp = requests.post(OT_URL, json={"query": SEARCH_Q, "variables": {"queryString": gene, "size": 5}})
    hits = resp.json().get("data", {}).get("search", {}).get("hits", [])
    for h in hits:
        if h.get("name", "").upper() == gene.upper() and h.get("entity") == "target":
            return h["id"]
    return hits[0]["id"] if hits else None

def ingest_gene(gene, limit=50):
    eid = resolve_ensembl(gene)
    if not eid:
        return []
    resp = requests.post(OT_URL, json={"query": TARGET_Q, "variables": {"ensemblId": eid, "size": limit}})
    data = resp.json().get("data", {}).get("target")
    if not data:
        return []
    rows = []
    symbol = data["approvedSymbol"]
    now = datetime.now(timezone.utc).isoformat()
    for assoc in data.get("associatedDiseases", {}).get("rows", []):
        disease = assoc["disease"]
        score = assoc.get("score", 0)
        rows.append({
            "record_id": f"{eid}_{disease['id']}",
            "source_key": "opentargets",
            "gene_symbol": symbol,
            "disease": disease.get("name"),
            "drug": None,
            "title": f"{symbol} - {disease.get('name')} (score: {score:.3f})",
            "summary": f"Association score {score:.3f}. Areas: {', '.join(ta['name'] for ta in disease.get('therapeuticAreas', []))}",
            "payload": json.dumps(assoc),
            "ingested_at": now,
            "source_updated_at": None,
        })
    tract = data.get("tractability") or []
    if tract:
        modalities = {}
        for t in tract:
            if t.get("value"):
                modalities.setdefault(t["modality"], []).append(t["label"])
        rows.append({
            "record_id": f"{eid}_tractability",
            "source_key": "opentargets",
            "gene_symbol": symbol,
            "disease": None, "drug": None,
            "title": f"{symbol} Druggability/Tractability",
            "summary": f"Tractable modalities: {json.dumps(modalities)}",
            "payload": json.dumps({"tractability": tract, "target_id": eid}),
            "ingested_at": now, "source_updated_at": None,
        })
    return rows

# ── Run ──
all_rows = []
for gene in TARGET_GENES:
    try:
        all_rows.extend(ingest_gene(gene, limit=50))
        print(f"  \u2705 {gene}")
    except Exception as e:
        print(f"  \u274C {gene}: {e}")

print(f"\nTotal: {len(all_rows)} records")

# ── Write ──
schema = StructType([
    StructField("record_id", StringType(), False),
    StructField("source_key", StringType(), False),
    StructField("gene_symbol", StringType(), True),
    StructField("disease", StringType(), True),
    StructField("drug", StringType(), True),
    StructField("title", StringType(), True),
    StructField("summary", StringType(), True),
    StructField("payload", StringType(), False),
    StructField("ingested_at", StringType(), False),
    StructField("source_updated_at", StringType(), True),
])

df = spark.createDataFrame(all_rows, schema=schema)
df = df.withColumn("ingested_at", F.to_timestamp("ingested_at")) \
       .withColumn("source_updated_at", F.to_timestamp("source_updated_at")) \
       .withColumn("payload", F.parse_json("payload"))
df.write.mode("overwrite").saveAsTable(TABLE)

count = spark.sql(f"SELECT COUNT(*) FROM {TABLE}").collect()[0][0]
print(f"\u2705 {TABLE}: {count} records")